# StressLab NGBoost Fabric 연구 학습
OneLake Gold를 Spark에서 읽고, 원본 NGBoost 학습 함수를 Fabric 드라이버의 전용 Python 환경에서 실행합니다.
각 실행의 모델, 구간·손실 확률 지표, 시나리오와 평가 행을 OneLake Files에 기록합니다.

In [ ]:
from pathlib import Path

LAKEHOUSE_ROOT = "abfss://2c303c0d-6c10-4988-91cb-856fe5f56071@onelake.dfs.fabric.microsoft.com/d6f926a1-b25d-41b5-a8cc-5961034475f4"
GOLD_FORMAT = "parquet"
GOLD_SOURCE = f"{LAKEHOUSE_ROOT}/Files/gold/gold_stress_daily_reconciled_v142_20261105/gold_stress_daily.parquet"
GOLD_DELTA_VERSION = None
DATASET_VERSION = "official20-reconciled-v142-20261105"
HORIZONS = (1, 5, 20)
ROUNDS = 300
CALIBRATION_START = "2024-01-01"
EVALUATION_START = "2025-01-01"
COMPARE_TUNED_E = True
EXPECTED_GOLD_ASSETS = 20
MAX_GOLD_ROWS = 250_000
OUTPUT_ROOT = f"{LAKEHOUSE_ROOT}/Files/stresslab/model_ngboost/runs"
SOURCE_SHA256 = "f6ba9a6e2e919f72002ffbc146dc7813b4f3291f1e8d5cc0531e4d563949156c"

for field, value in {"GOLD_SOURCE": GOLD_SOURCE, "OUTPUT_ROOT": OUTPUT_ROOT}.items():
    if not value.startswith(LAKEHOUSE_ROOT + "/"):
        raise ValueError(f"{field}: LH_StressLab_ML 경로를 지정하세요.")


In [ ]:
import base64
import hashlib
import io
import tempfile
import zipfile

root = Path(tempfile.mkdtemp(prefix="stresslab_ngboost_"))
source_root = root / "source"
bundle = base64.b64decode('...')
if hashlib.sha256(bundle).hexdigest() != SOURCE_SHA256:
    raise ValueError("학습 코드 SHA256 불일치")
with zipfile.ZipFile(io.BytesIO(bundle)) as archive:
    for member in archive.namelist():
        destination = source_root / member
        if not destination.resolve().is_relative_to(source_root.resolve()):
            raise ValueError("학습 코드 ZIP 경로 오류")
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(archive.read(member))
print("학습 코드 준비 완료")

In [ ]:
import json
import pandas as pd
from delta.tables import DeltaTable
from pyspark.sql import functions as F

old_timezone = spark.conf.get("spark.sql.session.timeZone")
try:
    spark.conf.set("spark.sql.session.timeZone", "UTC")
    selected_version = None
    reader = spark.read.format(GOLD_FORMAT)
    if GOLD_FORMAT == "delta":
        selected_version = GOLD_DELTA_VERSION
        if selected_version is None:
            selected_version = DeltaTable.forPath(spark, GOLD_SOURCE).history(1).first()["version"]
        reader = reader.option("versionAsOf", selected_version)
    elif GOLD_FORMAT != "parquet" or GOLD_DELTA_VERSION is not None:
        raise ValueError("Gold 형식은 delta 또는 parquet이며 Delta 버전은 delta에만 지정할 수 있습니다.")
    raw = reader.load(GOLD_SOURCE)
    schema = json.loads((source_root / "contracts/daily/schema.json").read_text(encoding="utf-8"))
    contract_columns = schema["items"]["required"]
    missing = sorted(set(contract_columns) - set(raw.columns))
    if missing:
        raise ValueError(f"Gold v1.1 계약 컬럼 누락: {missing}")
    extra_columns = sorted(set(raw.columns) - set(contract_columns))
    raw = raw.select(*contract_columns).where(F.col("dataset_version") == DATASET_VERSION)
    if any(field.dataType.simpleString() == "timestamp_ntz" for field in raw.schema.fields):
        raise ValueError("timestamp_ntz 컬럼의 원본 시간대를 확인하세요.")
    timestamp_columns = [field.name for field in raw.schema.fields
                         if field.dataType.simpleString() == "timestamp"]
    gold_pdf = raw.limit(MAX_GOLD_ROWS + 1).toPandas()
finally:
    spark.conf.set("spark.sql.session.timeZone", old_timezone)
if gold_pdf.empty or len(gold_pdf) > MAX_GOLD_ROWS:
    raise ValueError("Gold 버전에 행이 없거나 드라이버 수집 한도를 넘었습니다.")
for column in timestamp_columns:
    gold_pdf[column] = pd.to_datetime(gold_pdf[column], utc=True, errors="raise")
gold_path = root / "gold.parquet"
gold_pdf.to_parquet(gold_path, index=False)
print(f"Gold {DATASET_VERSION} · {len(gold_pdf):,}행 · Delta 버전 {selected_version}", flush=True)
print("계약 밖 추가 컬럼:", extra_columns, flush=True)

In [ ]:
import json
import os
import subprocess
import venv

env_dir = root / ".venv"
venv.EnvBuilder(with_pip=True).create(env_dir)
python = env_dir / "bin/python"
packages = ["ngboost==0.6.0", "pyarrow==26.0.0"]
if COMPARE_TUNED_E:
    packages.append("catboost==1.2.10")
driver_env = {**os.environ, "PYTHONNOUSERSITE": "1", "PYTHONDONTWRITEBYTECODE": "1",
              "OMP_NUM_THREADS": "1", "OPENBLAS_NUM_THREADS": "1"}
driver_env.pop("PYTHONPATH", None)
subprocess.run([str(python), "-m", "pip", "install", "--disable-pip-version-check",
                "--retries", "2", "--timeout", "25", *packages], check=True, env=driver_env)
subprocess.run([str(python), "-m", "pip", "check"], check=True, env=driver_env)

local_out = root / "run"
config = {
    "horizons": HORIZONS, "rounds": ROUNDS, "calibration_start": CALIBRATION_START,
    "evaluation_start": EVALUATION_START, "compare_tuned_e": COMPARE_TUNED_E, "baseline_dir": None,
    "source": {"location": GOLD_SOURCE, "format": GOLD_FORMAT, "dataset_version": DATASET_VERSION,
               "delta_version": selected_version, "projected_out_columns": extra_columns,
               "source_sha256": SOURCE_SHA256},
}
driver_code = '''
import json
import sys
from train import load_gold, run_experiment

gold = load_gold(sys.argv[1], dataset_version=sys.argv[3])
asset_count = gold.loc[~gold["is_mock"], "asset_id"].nunique()
if asset_count != int(sys.argv[4]):
    raise ValueError(f"Gold 종목 수 불일치: 실제 {asset_count}, 기대 {sys.argv[4]}")
info = run_experiment(gold, sys.argv[2], **json.loads(sys.argv[5]))
for horizon, summary in info["horizons"].items():
    print(horizon, "거래일 평가 표본", summary["test_rows"], "지원 종목", summary["assets"])
'''
command = [str(python), "-u", "-c", driver_code, str(gold_path), str(local_out),
           DATASET_VERSION, str(EXPECTED_GOLD_ASSETS), json.dumps(config)]
completed = subprocess.run(command, cwd=source_root / "analytics/model_ngboost", env=driver_env,
                           stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8")
if completed.returncode:
    raise RuntimeError(f"NGBoost 학습 프로세스 종료 코드 {completed.returncode}" + chr(10) + completed.stdout[-8000:])
print(completed.stdout, flush=True)

In [ ]:
from datetime import datetime, timezone
from uuid import uuid4
import notebookutils

run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid4().hex[:8]
destination = f"{OUTPUT_ROOT}/{run_id}"
if not notebookutils.fs.mkdirs(destination):
    raise RuntimeError(f"OneLake 실행 폴더 생성 실패: {destination}")
summary = local_out / "run_info.json"
if not summary.is_file():
    raise RuntimeError("완료된 학습 결과가 없습니다.")
for file in sorted(local_out.iterdir(), key=lambda path: (path == summary, path.name)):
    if not notebookutils.fs.cp(file.as_uri(), f"{destination}/{file.name}"):
        raise RuntimeError(f"OneLake 저장 실패: {file.name}")
print(f"완료: {destination}")
display(pd.read_csv(local_out / "metrics.csv").query("group == 'all' and coverage_target == 0.9"))